In [1]:
from langchain_core.prompts import PromptTemplate

# 질문과 답변을 포맷하는 프롬프트 템플릿 정의
example_prompt = PromptTemplate.from_template("질문: {question}\n답변: {answer}")

In [2]:
examples = [
    {
        "question": "주식 투자와 예금 중 어느 것이 더 수익률이 높은가?",
        "answer": """
후속 질문이 필요한가요: 네.
후속 질문: 주식 투자의 평균 수익률은 얼마인가요?
중간 답변: 주식 투자의 평균 수익률은 연 7%입니다.
후속 질문: 예금의 평균 이자율은 얼마인가요?
중간 답변: 예금의 평균 이자율은 연 1%입니다.
따라서 최종 답변은: 주식 투자
""",
    } ,
    {
        "question": "부동산과 채권 중 어느 것이 더 안정적인 투자처인가?",
        "answer": """
후속 질문이 필요한가요: 네.
후속 질문: 부동산 투자의 위험도는 어느 정도인가요?
중간 답변: 부동산 투자의 위험도는 중간 수준입니다.
후속 질문: 채권의 위험도는 어느 정도인가요?
중간 답변: 채권의 위험도는 낮은 편입니다.
따라서 최종 답변은: 채권
""",
    },
]

In [3]:
print(example_prompt.invoke(examples[0]).to_string())

질문: 주식 투자와 예금 중 어느 것이 더 수익률이 높은가?
답변: 
후속 질문이 필요한가요: 네.
후속 질문: 주식 투자의 평균 수익률은 얼마인가요?
중간 답변: 주식 투자의 평균 수익률은 연 7%입니다.
후속 질문: 예금의 평균 이자율은 얼마인가요?
중간 답변: 예금의 평균 이자율은 연 1%입니다.
따라서 최종 답변은: 주식 투자



In [4]:
# <`FewShotPromptTemplate` 를 이용한 퓨샷 프롬프트>

from langchain_core.prompts import FewShotPromptTemplate

# FewShotPromptTemplate 생성
prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    suffix="질문: {input}",
    input_variables=["input"]
)

# '부동산 투자' 주제로 프롬프트 호출 및 출력
print(
    prompt.invoke({"input": "부동산 투자의 장점은 무엇인가?"}).to_string()
)

질문: 주식 투자와 예금 중 어느 것이 더 수익률이 높은가?
답변: 
후속 질문이 필요한가요: 네.
후속 질문: 주식 투자의 평균 수익률은 얼마인가요?
중간 답변: 주식 투자의 평균 수익률은 연 7%입니다.
후속 질문: 예금의 평균 이자율은 얼마인가요?
중간 답변: 예금의 평균 이자율은 연 1%입니다.
따라서 최종 답변은: 주식 투자


질문: 부동산과 채권 중 어느 것이 더 안정적인 투자처인가?
답변: 
후속 질문이 필요한가요: 네.
후속 질문: 부동산 투자의 위험도는 어느 정도인가요?
중간 답변: 부동산 투자의 위험도는 중간 수준입니다.
후속 질문: 채권의 위험도는 어느 정도인가요?
중간 답변: 채권의 위험도는 낮은 편입니다.
따라서 최종 답변은: 채권


질문: 부동산 투자의 장점은 무엇인가?


In [24]:
# <예제 선택기를 이용한 퓨샷 프롬프트>

# 라이브러리 불러오기
from langchain_chroma import Chroma
from langchain_core.example_selectors import SemanticSimilarityExampleSelector
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
                model="text-embedding-nomic-embed-text-v1.5",
                base_url="http://...:.../v1",
                api_key="lm-studio",
                check_embedding_ctx_length=False,
            )

# 예제 선택기 초기화
example_selector = SemanticSimilarityExampleSelector.from_examples(
    examples,  # 사용할 예제 목록
    embeddings,  # 임베딩 생성에 사용되는 클래스
    Chroma,  # 임베딩을 저장하고 유사도 검색을 수행하는 벡터 스토어 클래스
    k=1,  # 선택할 예제의 수
)

# 입력과 가장 유사한 예제 선택
question = "부동산 투자의 장점은 무엇인가?"
selected_examples = example_selector.select_examples({"question": question})

# 선택된 예제 출력
print(f"입력과 가장 유사한 예제: {question}")
for example in selected_examples:
    print("\n")
    for k, v in example.items():
        print(f"{k}: {v}")

입력과 가장 유사한 예제: 부동산 투자의 장점은 무엇인가?


answer: 
후속 질문이 필요한가요: 네.
후속 질문: 부동산 투자의 위험도는 어느 정도인가요?
중간 답변: 부동산 투자의 위험도는 중간 수준입니다.
후속 질문: 채권의 위험도는 어느 정도인가요?
중간 답변: 채권의 위험도는 낮은 편입니다.
따라서 최종 답변은: 채권

question: 부동산과 채권 중 어느 것이 더 안정적인 투자처인가?


In [15]:
# <퓨샷 프롬프트 AI 모델 적용>

# 라이브러리 불러오기
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate
from langchain_openai import ChatOpenAI

# 예제 프롬프트 템플릿 생성
example_prompt = PromptTemplate(
    input_variables=["question", "answer"],
    template="질문: {question}\n답변: {answer}"
)

In [25]:
# 퓨샷 프롬프트 템플릿 설정
prompt = FewShotPromptTemplate(
    example_selector=example_selector,
    example_prompt=example_prompt,
    prefix="다음은 금융 관련 질문과 답변의 예시입니다:",
    suffix="질문: {input}\n답변:",
    input_variables=["input"]
)

In [35]:
model = ChatOpenAI( 
    # model="unsloth/gemma-4-e2b-it",
    # model="qwen3-30b-a3b-instruct-2507",
    model="qwen/qwen3.6-27b",
    base_url="http://...:.../v1",
    api_key="lm-studio",
)

In [36]:
# 체인 구성 및 실행
chain = prompt | model  # RunnableSequence를 사용하여 체인 연결

response = chain.invoke({"input": "부동산 투자의 장점은 무엇인가?"})  # invoke 메서드 사용
print(response.content)



답변: 
후속 질문이 필요한가요: 네.
후속 질문: 부동산 투자의 주요 수익 구조는 어떻게 되나요?
중간 답변: 임대 수익을 통한 지속적인 현금 흐름 창출과 장기 보유 시 발생하는 자산 가치 상승(자본 이득)으로 구성됩니다.
후속 질문: 거시경제적 및 투자 운용상의 장점은 무엇인가요?
중간 답변: 대출을 활용한 레버리지 효과로 투자 자본 대비 수익률을 높일 수 있으며, 물가 상승기에는 자산 가치가 함께 오르므로 인플레이션 헤지 효과가 있습니다. 또한, 주식 등 유가증권과 상관관계가 낮아 포트폴리오 위험 분산에 유리합니다.
따라서 최종 답변은: 부동산 투자의 주요 장점은 임대 수익과 자산 가치 상승을 통한 수익 창출, 레버리지 활용, 인플레이션 헤지 효과, 그리고 자산 포트폴리오의 다양화 및 안정성 제고입니다.


In [37]:
model = ChatOpenAI( 
    model="unsloth/gemma-4-e2b-it",
    base_url="http://...:.../v1",
    api_key="lm-studio",
)

In [38]:
# 체인 구성 및 실행
chain = prompt | model  # RunnableSequence를 사용하여 체인 연결

response = chain.invoke({"input": "부동산 투자의 장점은 무엇인가?"})  # invoke 메서드 사용
print(response.content)

부동산 투자의 장점은 다음과 같습니다:

* **자산 가치 상승 잠재력:** 시간이 지남에 따라 부동산 가치가 상승하여 자본 이득을 얻을 수 있습니다.
* **임대 수익:** 부동산을 통해 정기적인 임대 수익을 창출할 수 있습니다.
* **인플레이션 헤지:** 인플레이션 시기에 실물 자산인 부동산의 가치가 상승하여 화폐 가치 하락으로부터 자산을 보호하는 역할을 할 수 있습니다.
* **레버리지 활용 가능성:** 대출(레버리지)을 활용하여 적은 자기 자본으로도 큰 규모의 투자를 할 수 있습니다.

**후속 질문이 필요한가요:** 네. (어떤 측면에 대해 더 자세히 알고 싶으신가요? 예를 들어, 유동성이나 관리의 어려움 같은 단점도 궁금하신가요?)


In [39]:
model = ChatOpenAI( 
    model="qwen3-30b-a3b-instruct-2507",
    base_url="http://...:.../v1",
    api_key="lm-studio",
)

In [40]:
# 체인 구성 및 실행
chain = prompt | model  # RunnableSequence를 사용하여 체인 연결

response = chain.invoke({"input": "부동산 투자의 장점은 무엇인가?"})  # invoke 메서드 사용
print(response.content)

후속 질문이 필요한가요: 네.  
후속 질문: 부동산 투자의 수익 구조는 어떻게 되나요?  
중간 답변: 부동산 투자의 수익 구조는 임대 수입과 자산 가치 상승(자본 이득)으로 이루어집니다.  
후속 질문: 이러한 수익은 일정한가요,还 변동성이 크나요?  
중간 답변: 임대 수입은 비교적 안정적이지만, 시세 변동과 공실률 등에 따라 변동성이 있을 수 있습니다.  
따라서 최종 답변은: 부동산 투자의 장점은 일정한 임대수익과 자산 가치 상승을 통한 장기적인 재테크 효과가 있으며, 물적 자산으로서 인플레이션 대응 능력이 뛰어나다는 점입니다.
